# Sample-Path Dependent AIS Bounds for Approximate Dynamic Programming

This notebook implements and visualises the **sample-path dependent bound** from Theorem 2 (MDP Corollary) and compares it against the existing sup-norm and weighted-norm bounds from the model approximation paper.

**Key idea.** Instead of a single scalar bound on the suboptimality gap $V^{\hat\pi^\star}(s) - V^\star(s)$, we compute a *state-dependent* function $\alpha(s)$ such that

$$
2\,\alpha(s) \;\ge\; V^{\hat\pi^\star}(s) - V^\star(s) \qquad \forall\, s \in \mathcal{S}.
$$

Because $\alpha$ adapts to the local error structure, it can be **much tighter** than the uniform sup-norm or weighted-norm bounds in the operating region of the MDP.

## 1. Setup and Imports

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
from matplotlib import pyplot as plt

from models import InventoryMDP
from solvers import value_iteration, policy_evaluation
from bounds import (weight_function, compute_kappa,
                    weighted_norm_bound_alpha_beta, sample_path_bound)

## 2. Model Definition

We study an inventory MDP with state space $\mathcal{S} = \{-S_{\max},\ldots, S_{\max}\}$, action space $\mathcal{A} = \{0,\ldots,S_{\max}\}$, discount factor $\gamma = 0.75$, and Binomial$(n, q)$ demand.

| Parameter | True model $M$ | Approximate model $\hat M$ |
|---|---|---|
| $S_{\max}$ | 500 | 500 |
| $\gamma$ | 0.75 | 0.75 |
| Demand $\sim \text{Bin}(n,q)$ | $n=10,\; q=0.4$ | $n=10,\; q=0.5$ |
| Holding cost $c_h$ | 4 | 3.8 |
| Shortage cost $c_s$ | 2 | 2 |
| Ordering cost $p$ | 5 | 5 |

The per-step cost is
$$
c(s, a) = p\,a + h(s), \qquad h(s) = \begin{cases} c_h\, s & s \ge 0 \\ -c_s\, s & s < 0 \end{cases}.
$$

In [ ]:
M     = InventoryMDP(s_max=500, gamma=0.75, n=10, q=0.4, ch=4, cs=2, p=5)
M_hat = InventoryMDP(s_max=500, gamma=0.75, n=10, q=0.5, ch=3.8, cs=2, p=5)

print(f"State space size:  {M.num_states}")
print(f"Action space size: {M.num_actions}")
print(f"Demand support:    {{0, ..., {M.n}}}")

## 3. Value Iteration and Policy Evaluation

We solve for the optimal value functions and policies on both models using value iteration with the **H-array trick**, which avoids storing the full $(2S_{\max}+1) \times (S_{\max}+1) \times (2S_{\max}+1)$ transition matrix.

For any function $f$ over states, the expected value under transition $(s, a)$ reduces to:
$$
\sum_{s'} f(s')\, P(s'\mid s, a) = \sum_{w=0}^{n} W[w] \cdot f\bigl(\text{clip}(s + a - w)\bigr)
$$
where $W[w] = \Pr(D = w)$ is the demand PMF. We precompute the **H-array**:
$$
H[z] = \sum_{w=0}^{n} W[w] \cdot f\bigl(\text{clip}(z - w)\bigr), \qquad z \in \{-S_{\max}, \ldots, 2S_{\max}\}
$$
so that $\sum_{s'} f(s') P(s'\mid s,a) = H[s+a]$.

In [ ]:
print("Value iteration on M ...")
V_star, pi_star = value_iteration(M)

print("Value iteration on M_hat ...")
V_hat_star, pi_hat_star = value_iteration(M_hat)

# Base-stock levels
sigma_star = int(M.states[np.where(pi_star == 0)[0][0]])
sigma_hat  = int(M_hat.states[np.where(pi_hat_star == 0)[0][0]])
print(f"\nOptimal base-stock levels:")
print(f"  sigma*     = {sigma_star}  (true model)")
print(f"  sigma_hat* = {sigma_hat}  (approx model)")

In [ ]:
print("Policy evaluation: pi_hat_star on M ...")
V_pi_hat_star = policy_evaluation(M, pi_hat_star)

gap = V_pi_hat_star - V_star
print(f"\nSuboptimality gap  V^{{pi_hat*}} - V*:")
print(f"  max = {np.max(gap):.4f}")
print(f"  min = {np.min(gap):.6f}")

## 4. Existing Bounds: Sup-Norm and Weighted-Norm

The **model approximation bound** (Theorem 1 from the earlier paper) gives a uniform bound:

$$
\frac{V^{\hat\pi^\star}(s) - V^\star(s)}{w(s)} \;\le\; \frac{1}{1 - \gamma \kappa} \left( \left\| \frac{T^\star \hat V^\star - \hat T^\star \hat V^\star}{w} \right\|_\infty + \left\| \frac{T^{\hat\pi^\star} \hat V^\star - \hat T^{\hat\pi^\star} \hat V^\star}{w} \right\|_\infty \right)
$$

where $w(s) = 1 + \ell \cdot \hat h(s)$ is the weight function and $\kappa = \max_{\pi} \max_s \frac{\mathbb{E}_\pi[w(s')]}{w(s)}$.

- **Sup-norm** ($\ell = 0$): $w \equiv 1$, gives a single constant bound for all states.
- **Weighted-norm** ($\ell = 0.015$): state-dependent but conservative due to worst-case over all states in the weighted norm.

In [ ]:
states = M.states
s_max  = M.s_max

# --- Sup-norm bound (ell = 0) ---
w_sup = weight_function(M_hat, 0)
kappa_sup = max(
    compute_kappa(M, w_sup, pi_star),
    compute_kappa(M, w_sup, pi_hat_star),
    compute_kappa(M_hat, w_sup, pi_hat_star),
)
bound_sup_scalar = weighted_norm_bound_alpha_beta(
    V_hat_star, pi_hat_star, w_sup, M, M_hat, kappa_sup)
sup_bound_curve = bound_sup_scalar * w_sup

print(f"Sup-norm bound (ell=0):")
print(f"  kappa = {kappa_sup:.6f}")
print(f"  bound = {bound_sup_scalar:.4f}  (constant for all s)")

# --- Weighted-norm bound (ell = 0.015) ---
ell_w = 1.5e-2
w_w = weight_function(M_hat, ell_w)
kappa_w = max(
    compute_kappa(M, w_w, pi_star),
    compute_kappa(M, w_w, pi_hat_star),
    compute_kappa(M_hat, w_w, pi_hat_star),
)
bound_w_scalar = weighted_norm_bound_alpha_beta(
    V_hat_star, pi_hat_star, w_w, M, M_hat, kappa_w)
weighted_bound_curve = bound_w_scalar * w_w

print(f"\nWeighted-norm bound (ell={ell_w}):")
print(f"  kappa = {kappa_w:.6f}")
print(f"  scalar = {bound_w_scalar:.4f}")
print(f"  bound at s=0: {weighted_bound_curve[s_max]:.4f}")

## 5. Sample-Path Dependent Bound (Theorem 2)

The sample-path bound computes a state-dependent function $\alpha(s)$ satisfying $2\alpha(s) \ge V^{\hat\pi^\star}(s) - V^\star(s)$ via a **fixed-point iteration**. The key quantities are:

### 5.1 Cost Mismatch
$$
\varepsilon(s, a) = |c_M(s, a) - c_{\hat M}(s, a)| = |h_M(s) - h_{\hat M}(s)|
$$
Since $p$ is identical in both models, $\varepsilon$ depends only on $s$:
$$
\varepsilon(s) = \begin{cases} |c_h - \hat c_h| \cdot s = 0.2\, s & s \ge 0 \\ |c_s - \hat c_s| \cdot |s| = 0 & s < 0 \end{cases}
$$

### 5.2 Transition Mismatch
$$
\Delta(s, a) = \left| \sum_{s'} \hat V^\star(s')\, P_M(s'\mid s,a) - \sum_{s'} \hat V^\star(s')\, P_{\hat M}(s'\mid s,a) \right|
$$
Using H-arrays with the true ($W_M$) and approximate ($W_{\hat M}$) demand PMFs:
$$
\Delta(s, a) = |H_M[s+a] - H_{\hat M}[s+a]|
$$

### 5.3 Fixed-Point Iteration

Initialize $\alpha^0(s) = 0$. At each iteration $k$:

1. Build the propagation H-array (using **true** model dynamics):
$$
H_\alpha[z] = \sum_{w=0}^{n} W_M[w] \cdot \alpha^k\bigl(\text{clip}(z-w)\bigr)
$$

2. Compute the per-state-action error certificate:
$$
\beta(s, a) = \varepsilon(s) + \gamma \, H_\alpha[s+a] + \Delta(s, a)
$$

3. Update $\alpha$:
   - **$\alpha_{\max}$** (tighter): maximise over the two relevant policies only:
$$
\alpha^{k+1}_{\max}(s) = \max\!\big\{\beta(s, \pi^\star(s)),\; \beta(s, \hat\pi^\star(s))\big\}
$$
   - **$\alpha_{\sup}$** (looser): maximise over all actions:
$$
\alpha^{k+1}_{\sup}(s) = \max_{a \in \mathcal{A}} \beta(s, a)
$$

**Convergence** is guaranteed by the $\gamma$-contraction of the true model's transition operator. We iterate until $\max_s |\alpha^{k+1}(s) - \alpha^k(s)| < 10^{-10}$.

In [ ]:
alpha_max, alpha_sup, history = sample_path_bound(
    M, M_hat, V_hat_star, pi_star, pi_hat_star)

print(f"Fixed-point iteration converged in {len(history)} iterations")
print(f"max 2*alpha_max = {2 * np.max(alpha_max):.4f}")
print(f"max 2*alpha_sup = {2 * np.max(alpha_sup):.4f}")

### Validity Check

Verify that $2\,\alpha_{\max}(s) \ge V^{\hat\pi^\star}(s) - V^\star(s)$ for all $s$:

In [ ]:
slack = 2 * alpha_max - gap
print(f"min slack  = {np.min(slack):.6f}  (should be >= 0)")
print(f"Bound valid everywhere: {np.all(slack >= -1e-8)}")
print(f"alpha_max <= alpha_sup everywhere: {np.all(alpha_max <= alpha_sup + 1e-10)}")

## 6. Comparison Plots

### Plot 1: Convergence of the Fixed-Point Iteration

The iteration contracts at rate $\gamma = 0.75$, so $\max_s|\alpha^{k+1} - \alpha^k|$ decays geometrically.

In [ ]:
plt.rcParams['pdf.fonttype'] = 42
fig, ax = plt.subplots(figsize=(5, 3), dpi=150)

ax.semilogy(range(1, len(history) + 1), history, linewidth=1, color='#1f77b4')
ax.set_xlabel('Iteration $k$')
ax.set_ylabel(r'$\max_s |\alpha^{k+1}(s) - \alpha^k(s)|$')
ax.set_title('Fixed-point convergence', fontsize=9)
ax.grid(True)
plt.tight_layout()
plt.show()

### Plot 2: Shaded Region Comparison (Zoomed Out)

Three panels showing the gap between $V^{\hat\pi^\star}(s)$ and the lower bound $V^{\hat\pi^\star}(s) - \text{bound}(s)$ for each method. A **narrower shaded region** means a tighter bound.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3), dpi=150, sharey=True)

titles = ['Sup-norm', 'Weighted-norm', 'Sample-path']
bounds_list = [sup_bound_curve, weighted_bound_curve, 2 * alpha_max]
colors = ['#2ca02c', '#d62728', '#1f77b4']

for ax, title, bound, clr in zip(axes, titles, bounds_list, colors):
    ax.plot(states, V_pi_hat_star, linewidth=1, color='green',
            label=r'$V^{\hat\pi^\star}$')
    ax.plot(states, V_star, linewidth=1, color='black', linestyle='--',
            label=r'$V^\star$')
    ax.fill_between(states, V_pi_hat_star, V_pi_hat_star - bound,
                     alpha=0.25, color=clr)
    ax.set_title(title, fontsize=9)
    ax.set_xlabel('state')
    ax.grid(True)
    ax.set_xlim(-s_max, s_max)
    ax.set_ylim(-1000, 5000)

axes[0].set_ylabel('value')
axes[0].legend(loc='best', fontsize=6)
plt.tight_layout()
plt.show()

### Plot 3: Shaded Region Comparison (Zoomed In, $s \in [-10, 10]$)

In [ ]:
lo = s_max - 10
hi = s_max + 10 + 1

fig, axes = plt.subplots(1, 3, figsize=(12, 3), dpi=150, sharey=True)

for ax, title, bound, clr in zip(axes, titles, bounds_list, colors):
    ax.step(states[lo:hi], V_pi_hat_star[lo:hi], linewidth=1, color='green',
            label=r'$V^{\hat\pi^\star}$')
    ax.step(states[lo:hi], V_star[lo:hi], linewidth=1, color='black',
            linestyle='--', label=r'$V^\star$')
    ax.fill_between(states[lo:hi], V_pi_hat_star[lo:hi],
                     V_pi_hat_star[lo:hi] - bound[lo:hi],
                     alpha=0.25, color=clr, step='pre')
    ax.set_title(title, fontsize=9)
    ax.set_xlabel('state')
    ax.grid(True)

axes[0].set_ylabel('value')
axes[0].legend(loc='best', fontsize=6)
plt.tight_layout()
plt.show()

### Plot 4: Direct Bound Comparison (Zoomed, $s \in [-10, 10]$)

All bounds on a single axis: the true suboptimality gap $V^{\hat\pi^\star}(s) - V^\star(s)$ versus $2\alpha_{\max}$, $2\alpha_{\sup}$, the weighted-norm bound curve, and the sup-norm constant.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4), dpi=150)

ax.step(states[lo:hi], gap[lo:hi], linewidth=1.5, color='black',
        label=r'$V^{\hat\pi^\star} - V^\star$')
ax.step(states[lo:hi], 2 * alpha_max[lo:hi], linewidth=1, color='#1f77b4',
        label=r'$2\alpha_{\max}$ (sample-path)')
ax.step(states[lo:hi], 2 * alpha_sup[lo:hi], linewidth=1, color='#1f77b4',
        linestyle='--', label=r'$2\alpha_{\sup}$ (sample-path)')
ax.step(states[lo:hi], weighted_bound_curve[lo:hi], linewidth=1,
        color='#d62728', label='Weighted-norm')
ax.axhline(y=bound_sup_scalar, linewidth=1, color='#2ca02c',
           linestyle='-', label='Sup-norm')

ax.set_xlabel('state')
ax.set_ylabel('bound value')
ax.set_title('Bound comparison (operating region)', fontsize=9)
ax.legend(loc='best', fontsize=6)
ax.grid(True)
plt.tight_layout()
plt.show()

### Plot 5: $\alpha_{\max}$ vs $\alpha_{\sup}$

The tighter variant $\alpha_{\max}$ only maximises over the two policies $\pi^\star$ and $\hat\pi^\star$, while $\alpha_{\sup}$ maximises over all actions. By construction $\alpha_{\max}(s) \le \alpha_{\sup}(s)$.

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3), dpi=150)

ax.step(states[lo:hi], 2 * alpha_max[lo:hi], linewidth=1, color='#1f77b4',
        label=r'$2\alpha_{\max}$')
ax.step(states[lo:hi], 2 * alpha_sup[lo:hi], linewidth=1, color='#ff7f0e',
        label=r'$2\alpha_{\sup}$')

ax.set_xlabel('state')
ax.set_ylabel('bound value')
ax.set_title(r'$\alpha_{\max}$ vs $\alpha_{\sup}$', fontsize=9)
ax.legend(loc='best', fontsize=7)
ax.grid(True)
plt.tight_layout()
plt.show()

## 7. Error Decomposition

At the fixed point, $\alpha(s)$ decomposes as:
$$
\alpha(s) = \underbrace{\varepsilon(s,\hat\pi^\star(s))}_{\text{cost mismatch}} \;+\; \underbrace{\Delta(s,\hat\pi^\star(s))}_{\text{transition mismatch on }\hat V^\star} \;+\; \underbrace{\gamma \sum_{s'} \alpha(s')\, P_M(s'\mid s, \hat\pi^\star(s))}_{\text{propagated future error}}
$$

This decomposition reveals **which source of model mismatch dominates** at each state.

In [ ]:
# Compute the three components
epsilon = np.abs(M.h_vec(states) - M_hat.h_vec(states))

# H_delta: transition mismatch on V_hat_star
W_M  = M.W
W_Mh = M_hat.W
n_demand = len(W_M)
H_len = 3 * s_max + 1

H_M_v  = np.zeros(H_len)
H_Mh_v = np.zeros(H_len)
for z in range(-s_max, 2 * s_max + 1):
    for w in range(n_demand):
        ns = min(max(z - w, -s_max), s_max)
        H_M_v[z + s_max]  += W_M[w]  * V_hat_star[ns + s_max]
        H_Mh_v[z + s_max] += W_Mh[w] * V_hat_star[ns + s_max]
H_delta = np.abs(H_M_v - H_Mh_v)

# Delta at pi_hat_star
delta_at_pi_hat = np.zeros(M.num_states)
for i, s in enumerate(states):
    z_idx = min(int(s) + pi_hat_star[i], 2 * s_max) + s_max
    delta_at_pi_hat[i] = H_delta[z_idx]

# Propagated error: gamma * E[alpha_max(s') | s, pi_hat_star(s)]
H_alpha = np.zeros(H_len)
for z in range(-s_max, 2 * s_max + 1):
    for w in range(n_demand):
        ns = min(max(z - w, -s_max), s_max)
        H_alpha[z + s_max] += W_M[w] * alpha_max[ns + s_max]

propagated = np.zeros(M.num_states)
for i, s in enumerate(states):
    z_idx = min(int(s) + pi_hat_star[i], 2 * s_max) + s_max
    propagated[i] = M.gamma * H_alpha[z_idx]

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4), dpi=150)

ax.step(states[lo:hi], epsilon[lo:hi], linewidth=1, color='#1f77b4',
        label=r'$\varepsilon(s, \hat\pi^\star(s))$ — cost mismatch')
ax.step(states[lo:hi], delta_at_pi_hat[lo:hi], linewidth=1, color='#ff7f0e',
        label=r'$\Delta(s, \hat\pi^\star(s))$ — transition mismatch')
ax.step(states[lo:hi], propagated[lo:hi], linewidth=1, color='#2ca02c',
        label=r'$\gamma \sum_{s^\prime} \alpha(s^\prime) P(s^\prime|s,\hat\pi^\star(s))$ — propagated')

ax.set_xlabel('state')
ax.set_ylabel('error component')
ax.set_title('Error decomposition at fixed point', fontsize=9)
ax.legend(loc='best', fontsize=6)
ax.grid(True)
plt.tight_layout()
plt.show()

## 8. Summary Table

Comparison of the three bounding methods in the operating region $s \in \{-8, \ldots, 2\}$:

In [ ]:
op_lo = s_max - 8
op_hi = s_max + 2 + 1

print(f"{'Method':<25s} {'Max bound in [-8,2]':>20s} {'Max bound (all s)':>20s}")
print("-" * 67)
print(f"{'True gap':<25s} {np.max(gap[op_lo:op_hi]):>20.4f} {np.max(gap):>20.4f}")
print(f"{'2*alpha_max (sp)':<25s} {np.max(2*alpha_max[op_lo:op_hi]):>20.4f} {np.max(2*alpha_max):>20.4f}")
print(f"{'2*alpha_sup (sp)':<25s} {np.max(2*alpha_sup[op_lo:op_hi]):>20.4f} {np.max(2*alpha_sup):>20.4f}")
print(f"{'Weighted-norm':<25s} {np.max(weighted_bound_curve[op_lo:op_hi]):>20.4f} {np.max(weighted_bound_curve):>20.4f}")
print(f"{'Sup-norm':<25s} {bound_sup_scalar:>20.4f} {bound_sup_scalar:>20.4f}")
print()
print(f"Tightness ratio in [-8,2]:")
print(f"  sample-path / weighted = {np.max(2*alpha_max[op_lo:op_hi]) / np.max(weighted_bound_curve[op_lo:op_hi]):.2f}")
print(f"  sample-path / sup-norm = {np.max(2*alpha_max[op_lo:op_hi]) / bound_sup_scalar:.4f}")